# FlyRank ML Capstone: Content Opportunity Scoring Model

This notebook trains a predictive model to identify which content is most likely to decay in the upcoming 30 days based on historical Search Console signals.

In [1]:
import pandas as pd
import numpy as np
import sys

try:
    import duckdb
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, roc_auc_score, precision_score
except ImportError:
    print("Dependencies missing. Run: pip install duckdb scikit-learn pandas")
    sys.exit(0)


## 1. Data Extraction (DuckDB & Hugging Face)

In [2]:
con = duckdb.connect()
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
except:
    print("Warning: No Hugging Face token found. Executing with simulated local data for reproducibility.")
    # Simulated data if token fails
    np.random.seed(42)
    simulated_data = pd.DataFrame({
        'client_id': np.random.choice(['client_A', 'client_B'], 5000),
        'content_id': range(5000),
        'impressions_90d': np.random.randint(100, 10000, 5000),
        'gsc_avg_position': np.random.uniform(1, 50, 5000),
        'content_age_days': np.random.randint(10, 1000, 5000),
        'velocity_30d': np.random.uniform(0.5, 1.5, 5000),
        'clicks_30d': np.random.randint(0, 1000, 5000),
        'impressions_next_30d': np.random.randint(50, 5000, 5000),
        'month': np.random.choice(['2026-04', '2026-05', '2026-06'], 5000)
    })
    con.register('raw_data', simulated_data)
    REL = "raw_data"

df = con.sql(f"""
    SELECT *
    FROM {REL}
""").df()

print(f"Extracted {len(df)} rows for modeling.")


Extracted 5000 rows for modeling.


## 2. Feature Engineering & Label Definition

In [3]:
# Define Label: Is the page decaying? 
# (Impressions dropping by more than 20% compared to baseline)
baseline = df['impressions_90d'] / 3 # approximate 30d baseline
df['is_decaying'] = (df['impressions_next_30d'] < (baseline * 0.8)).astype(int)

features = ['impressions_90d', 'gsc_avg_position', 'content_age_days', 'velocity_30d', 'clicks_30d']
X = df[features]
y = df['is_decaying']

print("Label Distribution:\n", y.value_counts(normalize=True))


Label Distribution:
 is_decaying
0    0.7456
1    0.2544
Name: proportion, dtype: float64


## 3. Time-Aware Split (Leakage Prevention)

In [4]:
# Train on April/May, Test on June (Holdout)
train_mask = df['month'].isin(['2026-04', '2026-05'])
test_mask = df['month'] == '2026-06'

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Training instances: {len(X_train)}")
print(f"Testing instances: {len(X_test)}")


Training instances: 3326
Testing instances: 1674


## 4. Modeling & Results (Precision@50)

In [5]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Predict probabilities for the holdout set
preds_proba = model.predict_proba(X_test)[:, 1]

# Rank the top 50 pages most likely to decay
top_50_indices = np.argsort(preds_proba)[-50:][::-1]
top_50_true = y_test.iloc[top_50_indices]

precision_at_50 = top_50_true.mean()
auc = roc_auc_score(y_test, preds_proba)

print(f"Model ROC-AUC Score: {auc:.3f}")
print(f"Model Precision@50:  {precision_at_50*100:.1f}%")

# Compare to Baseline (Age-based: refresh oldest content)
baseline_top_50_indices = np.argsort(X_test['content_age_days'])[-50:][::-1]
baseline_precision = y_test.iloc[baseline_top_50_indices].mean()
print(f"Naive Age Baseline Precision@50: {baseline_precision*100:.1f}%")


Model ROC-AUC Score: 0.728
Model Precision@50:  38.0%
Naive Age Baseline Precision@50: 16.0%
